# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q causalinference pandas numpy matplotlib seaborn
    print('Packages installed.')

# Causal Inference: Propensity Score Matching (PSM)

In observational studies, we cannot randomly assign treatment. This leads to **selection bias**: the treated group might be fundamentally different from the control group.

**Propensity Score Matching (PSM)** attempts to mimic a randomized control trial (RCT) by matching treated units with control units that have similar probabilities of receiving the treatment (Propensity Scores).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from causalinference import CausalModel

np.random.seed(42)
print("Libraries loaded.")

## 1. Simulate Data with Selection Bias

Scenario: Does a **Job Training Program** (Treatment) increase **Income** (Outcome)?

**Confounder**: Motivation. Highly motivated people are:
1.  More likely to join the training.
2.  Likely to earn more anyway.

This creates an upward bias if we just compare means.

In [ ]:
n = 1000

# 1. Confounders
# Age: Older people earn more
age = np.random.normal(35, 10, n)

# Motivation (Unobserved in real life, but we use it to generate treatment)
motivation = np.random.normal(0, 1, n)

# 2. Treatment Assignment (Training)
# Probability of joining depends on Motivation and Age
prob_join = 1 / (1 + np.exp(-(0.5*motivation - 0.05*(age-35))))
treatment = np.random.binomial(1, prob_join)

# 3. Outcome (Income)
# Base income + Age effect + Motivation effect + TRUE TREATMENT EFFECT (2000) + Noise
income = 30000 + (500 * age) + (3000 * motivation) + (2000 * treatment) + np.random.normal(0, 2000, n)

df = pd.DataFrame({
    'Income': income,
    'Training': treatment,
    'Age': age,
    'Motivation': motivation # In a real PSM, we might try to use proxies for this
})

df.head()

## 2. Naive Comparison

Let's just compare the average income of those who took training vs those who didn't.

In [ ]:
naive_diff = df[df.Training == 1].Income.mean() - df[df.Training == 0].Income.mean()
print(f"Naive Difference: ${naive_diff:.2f}")
print(f"True Effect: $2000.00")
print(f"Bias: ${naive_diff - 2000:.2f}")

## 3. Propensity Score Matching

We use `CausalModel` to estimate the propensity score and match units.

*Note: We will assume 'Motivation' is observed for this demo. In reality, you can only match on observed variables.*

In [ ]:
Y = df['Income'].values
D = df['Training'].values
X = df[['Age', 'Motivation']].values

cm = CausalModel(Y, D, X)

# Estimate Propensity Score
cm.est_propensity_s()

# Estimate Average Treatment Effect (ATE) using Matching
cm.est_via_matching(matches=1, bias_adj=True)

print(cm.estimates)

## 4. Assessing Balance

A good match means the Treated and Control groups look similar on covariates *after* matching.

In [ ]:
print(cm.summary_stats)

## 5. Conclusion

The naive estimate was biased upwards because motivated people were more likely to take the training. By matching on Motivation and Age, the PSM estimate should be closer to the true effect ($2000).